In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Day1_Telematics") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

In [2]:
df_tele = spark.read.csv("/opt/spark-data/telematics.csv", header=True, inferSchema=True)
df_vehicle = spark.read.csv("/opt/spark-data/vehicles.csv", header=True, inferSchema=True)

In [4]:
df_tele.count()
df_tele.show(5)

+-------------------+----------+-----+-----------+----------+--------+--------+
|          timestamp|vehicle_id|speed|engine_temp|fuel_level|     lat|     lon|
+-------------------+----------+-----+-----------+----------+--------+--------+
|2024-03-01 00:00:00|        V1|   41|         84|        50|13.72701|77.92426|
|2024-03-01 00:00:00|        V2|   17|        105|        95|13.86915|77.90112|
|2024-03-01 00:00:00|        V3|   79|         94|        79|13.06656|78.44125|
|2024-03-01 00:00:00|        V4|  100|        117|        44|13.57825|78.28303|
|2024-03-01 00:00:00|        V5|   80|         70|        83|13.74648|77.68761|
+-------------------+----------+-----+-----------+----------+--------+--------+
only showing top 5 rows



In [6]:
result = df_tele.filter(df_tele["speed"] > 60) \
    .join(df_vehicle, "vehicle_id") \
    .groupBy("model") \
    .avg("speed")

In [7]:
result.show()

+-----+-----------------+
|model|       avg(speed)|
+-----+-----------------+
|  SUV|90.23410472183264|
|Sedan|90.63002586409593|
|Truck|90.74504777441156|
+-----+-----------------+



In [8]:
result.explain(True)

== Parsed Logical Plan ==
'Aggregate ['model], ['model, avg(speed#19) AS avg(speed)#199]
+- Project [vehicle_id#18, timestamp#17, speed#19, engine_temp#20, fuel_level#21, lat#22, lon#23, model#49, plant#50, year#51]
   +- Join Inner, (vehicle_id#18 = vehicle_id#48)
      :- Filter (speed#19 > 60)
      :  +- Relation [timestamp#17,vehicle_id#18,speed#19,engine_temp#20,fuel_level#21,lat#22,lon#23] csv
      +- Relation [vehicle_id#48,model#49,plant#50,year#51] csv

== Analyzed Logical Plan ==
model: string, avg(speed): double
Aggregate [model#49], [model#49, avg(speed#19) AS avg(speed)#199]
+- Project [vehicle_id#18, timestamp#17, speed#19, engine_temp#20, fuel_level#21, lat#22, lon#23, model#49, plant#50, year#51]
   +- Join Inner, (vehicle_id#18 = vehicle_id#48)
      :- Filter (speed#19 > 60)
      :  +- Relation [timestamp#17,vehicle_id#18,speed#19,engine_temp#20,fuel_level#21,lat#22,lon#23] csv
      +- Relation [vehicle_id#48,model#49,plant#50,year#51] csv

== Optimized Logical Pl